In [1]:
# ============================================================
# Ablation-Driven Evaluation of Trivial Feature Baselines
# for Realistic Graph Anomaly Detection
#
# Datasets used:
# - Bitcoin-Alpha
# - Bitcoin-OTC
# - FraudYelp / YelpChi
# - FraudAmazon
# - GADBench Reddit
#
# Excluded for now:
# - PyG Reddit
# - DGraph-Fin
#
# Saves:
# - logs/*.txt
# - results_json/*.json
# - summary/results_summary.csv
# - summary/results_summary.xlsx
# ============================================================

from __future__ import annotations

import os
import sys
import json
import time
import math
import pickle
import random
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    IsolationForest,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python:", sys.executable)
print("Working dir:", Path.cwd())
print("Seed:", SEED)

Python: D:\Users\ivan\anaconda3\envs\gad_dgl\python.exe
Working dir: C:\Users\ivan\WORK\workshops\ICDM\Ablation
Seed: 42


In [2]:
# ============================================================
# Paths
# ============================================================

DATA_ROOT = Path(r"C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets")

PATHS = {
    "bitcoin_alpha": DATA_ROOT / "bitcoin_snap" / "bitcoin_alpha_edges.csv",
    "bitcoin_otc": DATA_ROOT / "bitcoin_snap" / "bitcoin_otc_edges.csv",

    "fraud_yelp_graph": DATA_ROOT / "exports" / "fraud_yelp" / "fraud_yelp.bin",
    "fraud_yelp_features": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_feature.pt",
    "fraud_yelp_labels": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_label.pt",
    "fraud_yelp_train_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_train_mask.pt",
    "fraud_yelp_val_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_val_mask.pt",
    "fraud_yelp_test_mask": DATA_ROOT / "exports" / "fraud_yelp" / "nodes_review" / "fraud_yelp_review_ndata_test_mask.pt",

    "fraud_amazon_graph": DATA_ROOT / "exports" / "fraud_amazon" / "fraud_amazon.bin",
    "fraud_amazon_features": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_feature.pt",
    "fraud_amazon_labels": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_label.pt",
    "fraud_amazon_train_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_train_mask.pt",
    "fraud_amazon_val_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_val_mask.pt",
    "fraud_amazon_test_mask": DATA_ROOT / "exports" / "fraud_amazon" / "nodes_user" / "fraud_amazon_user_ndata_test_mask.pt",

    "gadbench_reddit": DATA_ROOT / "repos" / "GADBench-master" / "datasets" / "reddit",
}

OUT_ROOT = Path.cwd() / "ablation_trivial_baselines_outputs"
LOG_DIR = OUT_ROOT / "logs"
JSON_DIR = OUT_ROOT / "results_json"
SUMMARY_DIR = OUT_ROOT / "summary"
FEATURE_DIR = OUT_ROOT / "cached_features"

for d in [OUT_ROOT, LOG_DIR, JSON_DIR, SUMMARY_DIR, FEATURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUT_ROOT:", OUT_ROOT)

print("\nPath check:")
for k, p in PATHS.items():
    print(f"{k:30s} | {'OK' if p.exists() else 'MISSING'} | {p}")

DATA_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets
OUT_ROOT: C:\Users\ivan\WORK\workshops\ICDM\Ablation\ablation_trivial_baselines_outputs

Path check:
bitcoin_alpha                  | OK | C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_alpha_edges.csv
bitcoin_otc                    | OK | C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_otc_edges.csv
fraud_yelp_graph               | OK | C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\fraud_yelp.bin
fraud_yelp_features            | OK | C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_feature.pt
fraud_yelp_labels              | OK | C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\exports\fraud_yelp\nodes_review\fraud_yelp_review_ndata_label.pt
fraud_yelp_train_mask          | OK | C:\Users\ivan\WORK\worksho

In [4]:
# ============================================================
# Logging and saving helpers
# ============================================================

def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def safe_name(x: str) -> str:
    x = str(x)
    bad = ['\\', '/', ':', '*', '?', '"', '<', '>', '|', ' ', '(', ')', '[', ']', '{', '}', ',']
    for b in bad:
        x = x.replace(b, "_")
    while "__" in x:
        x = x.replace("__", "_")
    return x.strip("_")


def log_line(log_path: Path, msg: str, also_print: bool = True):
    line = f"[{now_str()}] {msg}"
    if also_print:
        print(line)
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def save_json(path: Path, obj: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    tmp.replace(path)


def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def tensor_to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def print_class_balance(y, prefix=""):
    y = np.asarray(y).astype(int)
    vals, counts = np.unique(y, return_counts=True)
    total = len(y)
    print(prefix + "class balance:")
    for v, c in zip(vals, counts):
        print(prefix + f"  class {v}: {c} ({c / total:.4f})")

In [5]:
# ============================================================
# Metrics
# ============================================================

def precision_recall_hits_at_k(
    y_true: np.ndarray,
    scores: np.ndarray,
    k: Optional[int] = None,
    anomaly_ratio: Optional[float] = None,
) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)

    n = len(y_true)
    n_pos = int(y_true.sum())

    if k is None:
        if anomaly_ratio is not None:
            k = max(1, int(round(n * anomaly_ratio)))
        else:
            k = max(1, n_pos)

    k = int(max(1, min(k, n)))

    order = np.argsort(-scores)
    top = order[:k]
    hits = int(y_true[top].sum())

    precision_at_k = hits / k
    recall_at_k = hits / max(1, n_pos)
    hits_at_k = float(hits > 0)

    return {
        "k": k,
        "hits": hits,
        "precision_at_k": float(precision_at_k),
        "recall_at_k": float(recall_at_k),
        "hits_at_k": float(hits_at_k),
    }


def compute_metrics(y_true: np.ndarray, scores: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)

    out = {}

    if len(np.unique(y_true)) < 2:
        out["roc_auc"] = None
        out["pr_auc"] = None
    else:
        out["roc_auc"] = float(roc_auc_score(y_true, scores))
        out["pr_auc"] = float(average_precision_score(y_true, scores))

    n_pos = int(y_true.sum())
    ratio = n_pos / max(1, len(y_true))

    for tag, k in [
        ("at_pos", n_pos),
        ("at_1pct", max(1, int(0.01 * len(y_true)))),
        ("at_5pct", max(1, int(0.05 * len(y_true)))),
    ]:
        d = precision_recall_hits_at_k(y_true, scores, k=k)
        for key, value in d.items():
            out[f"{key}_{tag}"] = value

    return out

In [6]:
# ============================================================
# Structural feature extraction
# No networkx required for main features.
#
# Features:
# - in_degree
# - out_degree
# - total_degree
# - weighted positive / negative in/out degree if signed
# - negative ratios
# - simple 2-hop approximation via neighbor degree statistics
# ============================================================

def build_node_index_from_edges(edges: pd.DataFrame) -> Tuple[Dict[Any, int], np.ndarray]:
    nodes = pd.Index(pd.concat([edges["source"], edges["target"]], ignore_index=True).unique())
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    return node_to_idx, nodes.values


def edge_df_to_indexed_arrays(edges: pd.DataFrame, node_to_idx: Dict[Any, int]):
    src = edges["source"].map(node_to_idx).to_numpy(dtype=np.int64)
    dst = edges["target"].map(node_to_idx).to_numpy(dtype=np.int64)
    return src, dst


def compute_structural_features_from_edge_df(
    edges: pd.DataFrame,
    n_nodes: Optional[int] = None,
    node_ids: Optional[np.ndarray] = None,
    source_col: str = "source",
    target_col: str = "target",
    weight_col: Optional[str] = None,
    signed_col: Optional[str] = None,
    dataset_name: str = "",
) -> Tuple[np.ndarray, List[str], np.ndarray]:
    """
    If node_ids is None, node ids are inferred from source/target.
    Returns X_struct, feature_names, node_ids.
    """

    df = edges[[source_col, target_col] + ([weight_col] if weight_col and weight_col in edges.columns else [])].copy()
    df = df.rename(columns={source_col: "source", target_col: "target"})

    if node_ids is None:
        node_to_idx, node_ids = build_node_index_from_edges(df)
    else:
        node_ids = np.asarray(node_ids)
        node_to_idx = {node: i for i, node in enumerate(node_ids)}

    if n_nodes is None:
        n_nodes = len(node_ids)

    src = df["source"].map(node_to_idx)
    dst = df["target"].map(node_to_idx)

    valid = src.notna() & dst.notna()
    src = src[valid].to_numpy(dtype=np.int64)
    dst = dst[valid].to_numpy(dtype=np.int64)

    if weight_col and weight_col in edges.columns:
        w = edges.loc[valid, weight_col].to_numpy(dtype=float)
    else:
        w = np.ones(len(src), dtype=float)

    # Basic degree features.
    out_deg = np.bincount(src, minlength=n_nodes).astype(float)
    in_deg = np.bincount(dst, minlength=n_nodes).astype(float)
    total_deg = in_deg + out_deg

    out_w_sum = np.bincount(src, weights=w, minlength=n_nodes).astype(float)
    in_w_sum = np.bincount(dst, weights=w, minlength=n_nodes).astype(float)

    abs_w = np.abs(w)
    out_abs_w_sum = np.bincount(src, weights=abs_w, minlength=n_nodes).astype(float)
    in_abs_w_sum = np.bincount(dst, weights=abs_w, minlength=n_nodes).astype(float)

    pos = (w > 0).astype(float)
    neg = (w < 0).astype(float)

    out_pos = np.bincount(src, weights=pos, minlength=n_nodes).astype(float)
    in_pos = np.bincount(dst, weights=pos, minlength=n_nodes).astype(float)
    out_neg = np.bincount(src, weights=neg, minlength=n_nodes).astype(float)
    in_neg = np.bincount(dst, weights=neg, minlength=n_nodes).astype(float)

    eps = 1e-9
    in_neg_ratio = in_neg / (in_pos + in_neg + eps)
    out_neg_ratio = out_neg / (out_pos + out_neg + eps)
    all_neg_ratio = (in_neg + out_neg) / (in_pos + out_pos + in_neg + out_neg + eps)

    # Neighbor degree aggregation.
    # For each node, aggregate degree of outgoing and incoming neighbors.
    neighbor_deg_sum_out = np.bincount(src, weights=total_deg[dst], minlength=n_nodes).astype(float)
    neighbor_deg_sum_in = np.bincount(dst, weights=total_deg[src], minlength=n_nodes).astype(float)

    avg_out_neighbor_deg = neighbor_deg_sum_out / (out_deg + eps)
    avg_in_neighbor_deg = neighbor_deg_sum_in / (in_deg + eps)
    avg_neighbor_deg = (neighbor_deg_sum_out + neighbor_deg_sum_in) / (total_deg + eps)

    # Simple local reciprocity.
    pair_set = set(zip(src.tolist(), dst.tolist()))
    reciprocal_flags = np.array([(b, a) in pair_set for a, b in zip(src, dst)], dtype=float)
    out_recip = np.bincount(src, weights=reciprocal_flags, minlength=n_nodes).astype(float)
    in_recip = np.bincount(dst, weights=reciprocal_flags, minlength=n_nodes).astype(float)
    reciprocity_ratio = (out_recip + in_recip) / (total_deg + eps)

    X = np.vstack([
        in_deg,
        out_deg,
        total_deg,
        np.log1p(in_deg),
        np.log1p(out_deg),
        np.log1p(total_deg),

        in_w_sum,
        out_w_sum,
        in_abs_w_sum,
        out_abs_w_sum,

        in_pos,
        out_pos,
        in_neg,
        out_neg,
        in_neg_ratio,
        out_neg_ratio,
        all_neg_ratio,

        avg_in_neighbor_deg,
        avg_out_neighbor_deg,
        avg_neighbor_deg,
        reciprocity_ratio,
    ]).T.astype(np.float32)

    names = [
        "in_degree",
        "out_degree",
        "total_degree",
        "log1p_in_degree",
        "log1p_out_degree",
        "log1p_total_degree",

        "in_weight_sum",
        "out_weight_sum",
        "in_abs_weight_sum",
        "out_abs_weight_sum",

        "in_pos_count",
        "out_pos_count",
        "in_neg_count",
        "out_neg_count",
        "in_neg_ratio",
        "out_neg_ratio",
        "all_neg_ratio",

        "avg_in_neighbor_degree",
        "avg_out_neighbor_degree",
        "avg_neighbor_degree",
        "reciprocity_ratio",
    ]

    return X, names, node_ids


def combine_relation_edge_csvs(edge_paths: List[Path]) -> pd.DataFrame:
    dfs = []
    for p in edge_paths:
        if not p.exists():
            print("[WARN] edge file missing:", p)
            continue
        df = pd.read_csv(p)
        if "source" not in df.columns or "target" not in df.columns:
            raise ValueError(f"Missing source/target in {p}")
        df = df[["source", "target"]].copy()
        df["weight"] = 1.0
        df["relation"] = p.parent.name
        dfs.append(df)
    if not dfs:
        raise RuntimeError("No edge CSVs loaded.")
    return pd.concat(dfs, ignore_index=True)

In [7]:
# ============================================================
# Dataset loaders
# ============================================================

def load_bitcoin_dataset(name: str, csv_path: Path, anomaly_quantile: float = 0.90) -> Dict[str, Any]:
    """
    Bitcoin datasets do not have official node anomaly labels.
    For this benchmark-style experiment we create a diagnostic pseudo-label:
    nodes in top anomaly_quantile by incoming negative trust ratio and negative in-degree are labeled anomalous.

    This is NOT real ground truth. Use it as shortcut/stress-test only.
    """

    log_path = LOG_DIR / f"dataset_load_{safe_name(name)}.txt"
    log_line(log_path, f"Loading {name} from {csv_path}")

    edges = pd.read_csv(csv_path)
    required = {"source", "target", "rating", "timestamp"}
    missing = required - set(edges.columns)
    if missing:
        raise ValueError(f"{name}: missing columns {missing}")

    X_struct, struct_names, node_ids = compute_structural_features_from_edge_df(
        edges,
        weight_col="rating",
        dataset_name=name,
    )

    # Pseudo-label from structural/signed anomaly signal.
    col = {n: i for i, n in enumerate(struct_names)}
    in_neg_ratio = X_struct[:, col["in_neg_ratio"]]
    in_neg_count = X_struct[:, col["in_neg_count"]]
    total_deg = X_struct[:, col["total_degree"]]

    raw_score = (
        2.0 * in_neg_ratio
        + np.log1p(in_neg_count)
        + 0.1 * np.log1p(total_deg)
    )

    threshold = np.quantile(raw_score, anomaly_quantile)
    y = (raw_score >= threshold).astype(int)

    # Avoid degenerate labels.
    if y.sum() < 5:
        top_k = max(5, int(0.05 * len(y)))
        order = np.argsort(-raw_score)
        y[:] = 0
        y[order[:top_k]] = 1

    # Split.
    idx = np.arange(len(y))
    train_idx, test_idx = train_test_split(
        idx,
        test_size=0.40,
        random_state=SEED,
        stratify=y if len(np.unique(y)) == 2 else None,
    )
    val_idx, test_idx = train_test_split(
        test_idx,
        test_size=0.50,
        random_state=SEED,
        stratify=y[test_idx] if len(np.unique(y[test_idx])) == 2 else None,
    )

    attr = None
    attr_names = []

    log_line(log_path, f"{name}: nodes={len(y)}, edges={len(edges)}, pseudo anomalies={int(y.sum())}")
    print_class_balance(y, prefix=f"{name} ")

    return {
        "name": name,
        "kind": "bitcoin_signed_graph_pseudo_labels",
        "node_ids": node_ids,
        "edges": edges,
        "X_attr": attr,
        "attr_names": attr_names,
        "X_struct": X_struct,
        "struct_names": struct_names,
        "y": y.astype(int),
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "notes": "Pseudo-labels from negative trust statistics; use as diagnostic stress-test, not real fraud ground truth.",
    }


def load_fraud_export_dataset(
    name: str,
    feature_path: Path,
    label_path: Path,
    train_mask_path: Path,
    val_mask_path: Path,
    test_mask_path: Path,
    edge_paths: List[Path],
    node_type: str,
) -> Dict[str, Any]:

    log_path = LOG_DIR / f"dataset_load_{safe_name(name)}.txt"
    log_line(log_path, f"Loading {name}")

    X_attr = torch.load(feature_path, map_location="cpu")
    y = torch.load(label_path, map_location="cpu")
    train_mask = torch.load(train_mask_path, map_location="cpu")
    val_mask = torch.load(val_mask_path, map_location="cpu")
    test_mask = torch.load(test_mask_path, map_location="cpu")

    X_attr = tensor_to_numpy(X_attr).astype(np.float32)
    y = tensor_to_numpy(y).astype(int).reshape(-1)
    train_mask = tensor_to_numpy(train_mask).astype(bool).reshape(-1)
    val_mask = tensor_to_numpy(val_mask).astype(bool).reshape(-1)
    test_mask = tensor_to_numpy(test_mask).astype(bool).reshape(-1)

    n_nodes = len(y)
    node_ids = np.arange(n_nodes)

    edges = combine_relation_edge_csvs(edge_paths)

    X_struct, struct_names, _ = compute_structural_features_from_edge_df(
        edges,
        n_nodes=n_nodes,
        node_ids=node_ids,
        weight_col="weight",
        dataset_name=name,
    )

    train_idx = np.where(train_mask)[0]
    val_idx = np.where(val_mask)[0]
    test_idx = np.where(test_mask)[0]

    # Fallback if masks are empty/bad.
    if len(train_idx) == 0 or len(test_idx) == 0:
        idx = np.arange(len(y))
        train_idx, test_idx = train_test_split(
            idx,
            test_size=0.40,
            random_state=SEED,
            stratify=y if len(np.unique(y)) == 2 else None,
        )
        val_idx, test_idx = train_test_split(
            test_idx,
            test_size=0.50,
            random_state=SEED,
            stratify=y[test_idx] if len(np.unique(y[test_idx])) == 2 else None,
        )

    attr_names = [f"attr_{i}" for i in range(X_attr.shape[1])]

    log_line(log_path, f"{name}: nodes={n_nodes}, edges={len(edges)}, anomalies={int(y.sum())}")
    log_line(log_path, f"{name}: X_attr={X_attr.shape}, X_struct={X_struct.shape}")
    print_class_balance(y, prefix=f"{name} ")

    return {
        "name": name,
        "kind": "dgl_fraud_heterograph",
        "node_ids": node_ids,
        "edges": edges,
        "X_attr": X_attr,
        "attr_names": attr_names,
        "X_struct": X_struct,
        "struct_names": struct_names,
        "y": y,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "notes": "Real labels from DGL fraud benchmark.",
    }


def load_gadbench_reddit(path: Path) -> Optional[Dict[str, Any]]:
    """
    Flexible loader for GADBench Reddit file.
    If format is not recognized, returns None and prints keys/types.
    """

    name = "GADBench_Reddit"
    log_path = LOG_DIR / f"dataset_load_{safe_name(name)}.txt"
    log_line(log_path, f"Loading {name} from {path}")

    if not path.exists():
        log_line(log_path, "GADBench Reddit file missing.")
        return None

    obj = None
    errors = []

    try:
        obj = torch.load(path, map_location="cpu")
        log_line(log_path, "Loaded by torch.load")
    except Exception as e:
        errors.append(f"torch.load failed: {repr(e)}")

    if obj is None:
        try:
            with open(path, "rb") as f:
                obj = pickle.load(f)
            log_line(log_path, "Loaded by pickle.load")
        except Exception as e:
            errors.append(f"pickle.load failed: {repr(e)}")

    if obj is None:
        for e in errors:
            log_line(log_path, e)
        return None

    log_line(log_path, f"Object type: {type(obj)}")

    # Try common cases.
    if isinstance(obj, dict):
        keys = list(obj.keys())
        log_line(log_path, f"Dict keys: {keys}")

        for k, v in obj.items():
            if hasattr(v, "shape"):
                log_line(log_path, f"  {k}: shape={v.shape}, type={type(v)}")
            else:
                log_line(log_path, f"  {k}: type={type(v)}")

        possible_x = ["x", "X", "features", "feature", "feat", "attrs"]
        possible_y = ["y", "Y", "labels", "label", "target"]
        possible_edge = ["edge_index", "edges", "edge", "adj"]

        X_attr = None
        y = None
        edge_index = None

        for k in possible_x:
            if k in obj:
                X_attr = tensor_to_numpy(obj[k]).astype(np.float32)
                break

        for k in possible_y:
            if k in obj:
                y = tensor_to_numpy(obj[k]).astype(int).reshape(-1)
                break

        for k in possible_edge:
            if k in obj:
                edge_index = tensor_to_numpy(obj[k])
                break

        if X_attr is None or y is None:
            log_line(log_path, "Could not identify features/labels. Skipping GADBench Reddit.")
            return None

        n_nodes = len(y)

        # Convert edge_index to edge DataFrame if available.
        if edge_index is not None:
            edge_index = np.asarray(edge_index)
            if edge_index.shape[0] == 2:
                src = edge_index[0]
                dst = edge_index[1]
            elif edge_index.shape[1] == 2:
                src = edge_index[:, 0]
                dst = edge_index[:, 1]
            else:
                src, dst = np.array([], dtype=int), np.array([], dtype=int)

            edges = pd.DataFrame({
                "source": src.astype(int),
                "target": dst.astype(int),
                "weight": 1.0,
                "relation": "reddit",
            })
        else:
            edges = pd.DataFrame(columns=["source", "target", "weight", "relation"])

        if len(edges) > 0:
            X_struct, struct_names, _ = compute_structural_features_from_edge_df(
                edges,
                n_nodes=n_nodes,
                node_ids=np.arange(n_nodes),
                weight_col="weight",
                dataset_name=name,
            )
        else:
            X_struct = np.zeros((n_nodes, 1), dtype=np.float32)
            struct_names = ["dummy_struct_zero"]

        idx = np.arange(n_nodes)
        train_idx, test_idx = train_test_split(
            idx,
            test_size=0.40,
            random_state=SEED,
            stratify=y if len(np.unique(y)) == 2 else None,
        )
        val_idx, test_idx = train_test_split(
            test_idx,
            test_size=0.50,
            random_state=SEED,
            stratify=y[test_idx] if len(np.unique(y[test_idx])) == 2 else None,
        )

        attr_names = [f"attr_{i}" for i in range(X_attr.shape[1])]

        log_line(log_path, f"{name}: nodes={n_nodes}, edges={len(edges)}, anomalies={int(y.sum())}")
        print_class_balance(y, prefix=f"{name} ")

        return {
            "name": name,
            "kind": "gadbench_reddit",
            "node_ids": np.arange(n_nodes),
            "edges": edges,
            "X_attr": X_attr,
            "attr_names": attr_names,
            "X_struct": X_struct,
            "struct_names": struct_names,
            "y": y,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "notes": "Loaded from GADBench file.",
        }

    # PyG-like object.
    if hasattr(obj, "x") and hasattr(obj, "y"):
        X_attr = tensor_to_numpy(obj.x).astype(np.float32)
        y = tensor_to_numpy(obj.y).astype(int).reshape(-1)
        n_nodes = len(y)

        if hasattr(obj, "edge_index"):
            edge_index = tensor_to_numpy(obj.edge_index)
            src = edge_index[0]
            dst = edge_index[1]
            edges = pd.DataFrame({
                "source": src.astype(int),
                "target": dst.astype(int),
                "weight": 1.0,
                "relation": "reddit",
            })
        else:
            edges = pd.DataFrame(columns=["source", "target", "weight", "relation"])

        if len(edges) > 0:
            X_struct, struct_names, _ = compute_structural_features_from_edge_df(
                edges,
                n_nodes=n_nodes,
                node_ids=np.arange(n_nodes),
                weight_col="weight",
                dataset_name=name,
            )
        else:
            X_struct = np.zeros((n_nodes, 1), dtype=np.float32)
            struct_names = ["dummy_struct_zero"]

        idx = np.arange(n_nodes)
        train_idx, test_idx = train_test_split(
            idx,
            test_size=0.40,
            random_state=SEED,
            stratify=y if len(np.unique(y)) == 2 else None,
        )
        val_idx, test_idx = train_test_split(
            test_idx,
            test_size=0.50,
            random_state=SEED,
            stratify=y[test_idx] if len(np.unique(y[test_idx])) == 2 else None,
        )

        attr_names = [f"attr_{i}" for i in range(X_attr.shape[1])]

        return {
            "name": name,
            "kind": "gadbench_reddit",
            "node_ids": np.arange(n_nodes),
            "edges": edges,
            "X_attr": X_attr,
            "attr_names": attr_names,
            "X_struct": X_struct,
            "struct_names": struct_names,
            "y": y,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "notes": "Loaded from PyG-like GADBench object.",
        }

    log_line(log_path, "Unknown object format. Skipping GADBench Reddit.")
    return None

In [8]:
# ============================================================
# Load all selected datasets
# ============================================================

datasets = []

# Bitcoin Alpha / OTC.
if PATHS["bitcoin_alpha"].exists():
    datasets.append(load_bitcoin_dataset("Bitcoin_Alpha", PATHS["bitcoin_alpha"], anomaly_quantile=0.90))

if PATHS["bitcoin_otc"].exists():
    datasets.append(load_bitcoin_dataset("Bitcoin_OTC", PATHS["bitcoin_otc"], anomaly_quantile=0.90))

# FraudYelp.
fraud_yelp_edges = [
    DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rsr_review" / "fraud_yelp_review_net_rsr_review_edges.csv",
    DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rtr_review" / "fraud_yelp_review_net_rtr_review_edges.csv",
    DATA_ROOT / "exports" / "fraud_yelp" / "edges_review_net_rur_review" / "fraud_yelp_review_net_rur_review_edges.csv",
]

if PATHS["fraud_yelp_features"].exists() and PATHS["fraud_yelp_labels"].exists():
    datasets.append(load_fraud_export_dataset(
        name="FraudYelp",
        feature_path=PATHS["fraud_yelp_features"],
        label_path=PATHS["fraud_yelp_labels"],
        train_mask_path=PATHS["fraud_yelp_train_mask"],
        val_mask_path=PATHS["fraud_yelp_val_mask"],
        test_mask_path=PATHS["fraud_yelp_test_mask"],
        edge_paths=fraud_yelp_edges,
        node_type="review",
    ))

# FraudAmazon.
fraud_amazon_edges = [
    DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_upu_user" / "fraud_amazon_user_net_upu_user_edges.csv",
    DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_usu_user" / "fraud_amazon_user_net_usu_user_edges.csv",
    DATA_ROOT / "exports" / "fraud_amazon" / "edges_user_net_uvu_user" / "fraud_amazon_user_net_uvu_user_edges.csv",
]

if PATHS["fraud_amazon_features"].exists() and PATHS["fraud_amazon_labels"].exists():
    datasets.append(load_fraud_export_dataset(
        name="FraudAmazon",
        feature_path=PATHS["fraud_amazon_features"],
        label_path=PATHS["fraud_amazon_labels"],
        train_mask_path=PATHS["fraud_amazon_train_mask"],
        val_mask_path=PATHS["fraud_amazon_val_mask"],
        test_mask_path=PATHS["fraud_amazon_test_mask"],
        edge_paths=fraud_amazon_edges,
        node_type="user",
    ))

# GADBench Reddit.
gad_reddit = load_gadbench_reddit(PATHS["gadbench_reddit"])
if gad_reddit is not None:
    datasets.append(gad_reddit)

print("\n" + "=" * 120)
print("LOADED DATASETS")
print("=" * 120)
for ds in datasets:
    print(ds["name"], "| kind:", ds["kind"], "| nodes:", len(ds["y"]), "| anomalies:", int(np.sum(ds["y"])))

[2026-05-17 09:20:28] Loading Bitcoin_Alpha from C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_alpha_edges.csv
[2026-05-17 09:20:28] Bitcoin_Alpha: nodes=3783, edges=24186, pseudo anomalies=383
Bitcoin_Alpha class balance:
Bitcoin_Alpha   class 0: 3400 (0.8988)
Bitcoin_Alpha   class 1: 383 (0.1012)
[2026-05-17 09:20:28] Loading Bitcoin_OTC from C:\Users\ivan\WORK\workshops\ICDM\Ablation\graph_anomaly_datasets\bitcoin_snap\bitcoin_otc_edges.csv
[2026-05-17 09:20:29] Bitcoin_OTC: nodes=5881, edges=35592, pseudo anomalies=591
Bitcoin_OTC class balance:
Bitcoin_OTC   class 0: 5290 (0.8995)
Bitcoin_OTC   class 1: 591 (0.1005)
[2026-05-17 09:20:29] Loading FraudYelp
[2026-05-17 09:20:48] FraudYelp: nodes=45954, edges=8051348, anomalies=6677
[2026-05-17 09:20:48] FraudYelp: X_attr=(45954, 32), X_struct=(45954, 21)
FraudYelp class balance:
FraudYelp   class 0: 39277 (0.8547)
FraudYelp   class 1: 6677 (0.1453)
[2026-05-17 09:20:48] Loading FraudAmazon
[2

In [9]:
# ============================================================
# Ablation feature construction
# ============================================================

def get_feature_matrix_for_ablation(ds: Dict[str, Any], ablation: str) -> Tuple[np.ndarray, List[str]]:
    X_attr = ds.get("X_attr", None)
    X_struct = ds.get("X_struct", None)
    attr_names = ds.get("attr_names", [])
    struct_names = ds.get("struct_names", [])

    if X_struct is None:
        X_struct = np.zeros((len(ds["y"]), 1), dtype=np.float32)
        struct_names = ["dummy_struct_zero"]

    if X_attr is None:
        X_attr = np.zeros((len(ds["y"]), 0), dtype=np.float32)
        attr_names = []

    if ablation == "attr_only":
        if X_attr.shape[1] == 0:
            # Fallback to zero feature; this will make model mostly useless but keeps pipeline consistent.
            return np.zeros((len(ds["y"]), 1), dtype=np.float32), ["dummy_no_attr"]
        return X_attr.astype(np.float32), attr_names

    if ablation == "struct_only":
        return X_struct.astype(np.float32), struct_names

    if ablation == "full":
        X = np.concatenate([X_attr, X_struct], axis=1).astype(np.float32)
        names = attr_names + struct_names
        return X, names

    if ablation == "degree_only":
        degree_cols = []
        degree_names = []
        for i, n in enumerate(struct_names):
            if "degree" in n or "deg" in n:
                degree_cols.append(i)
                degree_names.append(n)

        if not degree_cols:
            return np.zeros((len(ds["y"]), 1), dtype=np.float32), ["dummy_no_degree"]

        return X_struct[:, degree_cols].astype(np.float32), degree_names

    if ablation == "no_degree_struct":
        keep_cols = []
        keep_names = []
        for i, n in enumerate(struct_names):
            if "degree" not in n and "deg" not in n:
                keep_cols.append(i)
                keep_names.append(n)

        if not keep_cols:
            return np.zeros((len(ds["y"]), 1), dtype=np.float32), ["dummy_no_non_degree_struct"]

        return X_struct[:, keep_cols].astype(np.float32), keep_names

    raise ValueError(f"Unknown ablation: {ablation}")


ABLATIONS = [
    "attr_only",
    "struct_only",
    "full",
    "degree_only",
    "no_degree_struct",
]

print("Ablations:", ABLATIONS)

Ablations: ['attr_only', 'struct_only', 'full', 'degree_only', 'no_degree_struct']


In [10]:
# ============================================================
# Models
# ============================================================

def make_supervised_model(model_name: str):
    if model_name == "LogisticRegression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="lbfgs",
                random_state=SEED,
            )),
        ])

    if model_name == "RandomForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=SEED,
            )),
        ])

    if model_name == "ExtraTrees":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", ExtraTreesClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED,
            )),
        ])

    if model_name == "GradientBoosting":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", GradientBoostingClassifier(
                n_estimators=150,
                learning_rate=0.05,
                max_depth=3,
                random_state=SEED,
            )),
        ])

    raise ValueError(model_name)


def make_unsupervised_model(model_name: str, contamination: float):
    if model_name == "IsolationForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", IsolationForest(
                n_estimators=300,
                contamination=max(1e-4, min(0.49, contamination)),
                random_state=SEED,
                n_jobs=-1,
            )),
        ])

    raise ValueError(model_name)


SUPERVISED_MODELS = [
    "LogisticRegression",
    "RandomForest",
    "ExtraTrees",
    "GradientBoosting",
]

UNSUPERVISED_MODELS = [
    "IsolationForest",
]

print("Supervised models:", SUPERVISED_MODELS)
print("Unsupervised models:", UNSUPERVISED_MODELS)

Supervised models: ['LogisticRegression', 'RandomForest', 'ExtraTrees', 'GradientBoosting']
Unsupervised models: ['IsolationForest']


In [11]:
# ============================================================
# Run one experiment with resume
# ============================================================

def result_paths(dataset_name: str, ablation: str, model_name: str) -> Tuple[Path, Path]:
    run_id = f"{safe_name(dataset_name)}__{safe_name(ablation)}__{safe_name(model_name)}"
    json_path = JSON_DIR / f"{run_id}.json"
    txt_path = LOG_DIR / f"{run_id}.txt"
    return json_path, txt_path


def fit_predict_scores(
    model_name: str,
    X: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    y = y.astype(int)

    X_train = X[train_idx]
    y_train = y[train_idx]
    X_test = X[test_idx]

    info = {}

    if model_name in SUPERVISED_MODELS:
        model = make_supervised_model(model_name)
        model.fit(X_train, y_train)

        clf = model.named_steps["clf"]

        if hasattr(clf, "predict_proba"):
            scores = model.predict_proba(X_test)[:, 1]
        elif hasattr(clf, "decision_function"):
            scores = model.decision_function(X_test)
        else:
            pred = model.predict(X_test)
            scores = pred.astype(float)

        info["mode"] = "supervised"

    elif model_name in UNSUPERVISED_MODELS:
        contamination = float(max(1e-4, min(0.49, y_train.mean() if y_train.mean() > 0 else 0.05)))
        model = make_unsupervised_model(model_name, contamination=contamination)

        # For unsupervised, fit only on training X, ignoring labels.
        model.fit(X_train)

        clf = model.named_steps["clf"]

        # IsolationForest: negative score means more anomalous.
        raw = clf.decision_function(model.named_steps["imputer"].transform(X_test))
        scores = -raw

        info["mode"] = "unsupervised"
        info["contamination"] = contamination

    else:
        raise ValueError(model_name)

    return np.asarray(scores, dtype=float), info


def run_experiment(
    ds: Dict[str, Any],
    ablation: str,
    model_name: str,
    force: bool = False,
) -> Dict[str, Any]:

    dataset_name = ds["name"]
    json_path, txt_path = result_paths(dataset_name, ablation, model_name)

    if json_path.exists() and not force:
        print(f"[SKIP] existing: {json_path.name}")
        return load_json(json_path)

    log_line(txt_path, "=" * 100)
    log_line(txt_path, f"RUN dataset={dataset_name}, ablation={ablation}, model={model_name}")
    log_line(txt_path, "=" * 100)

    t0 = time.time()

    X, feature_names = get_feature_matrix_for_ablation(ds, ablation)
    y = ds["y"].astype(int)
    train_idx = ds["train_idx"]
    val_idx = ds["val_idx"]
    test_idx = ds["test_idx"]

    # Train on train+val for final test evaluation.
    train_full_idx = np.unique(np.concatenate([train_idx, val_idx]))

    log_line(txt_path, f"X shape: {X.shape}")
    log_line(txt_path, f"n features: {len(feature_names)}")
    log_line(txt_path, f"train+val size: {len(train_full_idx)}")
    log_line(txt_path, f"test size: {len(test_idx)}")
    log_line(txt_path, f"total anomalies: {int(y.sum())}")
    log_line(txt_path, f"train anomalies: {int(y[train_full_idx].sum())}")
    log_line(txt_path, f"test anomalies: {int(y[test_idx].sum())}")

    if len(np.unique(y[train_full_idx])) < 2 and model_name in SUPERVISED_MODELS:
        log_line(txt_path, "Only one class in training split. Skipping supervised model.")
        result = {
            "dataset": dataset_name,
            "dataset_kind": ds.get("kind"),
            "ablation": ablation,
            "model": model_name,
            "status": "skipped_one_class_train",
            "metrics": {},
            "runtime_sec": time.time() - t0,
        }
        save_json(json_path, result)
        return result

    if len(np.unique(y[test_idx])) < 2:
        log_line(txt_path, "Only one class in test split. Metrics may be undefined.")

    try:
        scores, model_info = fit_predict_scores(
            model_name=model_name,
            X=X,
            y=y,
            train_idx=train_full_idx,
            test_idx=test_idx,
        )

        metrics = compute_metrics(y[test_idx], scores)

        runtime = time.time() - t0

        result = {
            "dataset": dataset_name,
            "dataset_kind": ds.get("kind"),
            "dataset_notes": ds.get("notes"),
            "ablation": ablation,
            "model": model_name,
            "status": "ok",
            "n_nodes": int(len(y)),
            "n_features": int(X.shape[1]),
            "n_train": int(len(train_full_idx)),
            "n_test": int(len(test_idx)),
            "n_anomalies_total": int(y.sum()),
            "n_anomalies_train": int(y[train_full_idx].sum()),
            "n_anomalies_test": int(y[test_idx].sum()),
            "feature_names": feature_names,
            "model_info": model_info,
            "metrics": metrics,
            "runtime_sec": runtime,
            "created_at": now_str(),
        }

        save_json(json_path, result)

        log_line(txt_path, "METRICS:")
        for k, v in metrics.items():
            log_line(txt_path, f"  {k}: {v}")

        log_line(txt_path, f"Runtime sec: {runtime:.2f}")
        log_line(txt_path, f"Saved JSON: {json_path}")

        return result

    except Exception as e:
        runtime = time.time() - t0

        log_line(txt_path, f"[ERROR] {repr(e)}")

        result = {
            "dataset": dataset_name,
            "dataset_kind": ds.get("kind"),
            "ablation": ablation,
            "model": model_name,
            "status": "error",
            "error": repr(e),
            "runtime_sec": runtime,
            "created_at": now_str(),
        }

        save_json(json_path, result)
        return result

In [12]:
# ============================================================
# Run all experiments
# Resume-safe.
# ============================================================

ALL_MODELS = SUPERVISED_MODELS + UNSUPERVISED_MODELS

master_log = LOG_DIR / "MASTER_RUN_LOG.txt"
log_line(master_log, "=" * 120)
log_line(master_log, "STARTING ABLATION-DRIVEN TRIVIAL BASELINE EVALUATION")
log_line(master_log, "=" * 120)

all_results = []

for ds in datasets:
    dataset_name = ds["name"]

    log_line(master_log, f"DATASET: {dataset_name}")
    log_line(master_log, f"  kind: {ds.get('kind')}")
    log_line(master_log, f"  nodes: {len(ds['y'])}")
    log_line(master_log, f"  anomalies: {int(np.sum(ds['y']))}")

    for ablation in ABLATIONS:
        for model_name in ALL_MODELS:
            print("\n" + "=" * 120)
            print(f"RUNNING: dataset={dataset_name}, ablation={ablation}, model={model_name}")
            print("=" * 120)

            res = run_experiment(
                ds=ds,
                ablation=ablation,
                model_name=model_name,
                force=False,
            )
            all_results.append(res)

log_line(master_log, "FINISHED ALL REQUESTED RUNS")

[2026-05-17 09:22:18] ========================================================================================================================
[2026-05-17 09:22:18] STARTING ABLATION-DRIVEN TRIVIAL BASELINE EVALUATION
[2026-05-17 09:22:18] ========================================================================================================================
[2026-05-17 09:22:18] DATASET: Bitcoin_Alpha
[2026-05-17 09:22:18]   kind: bitcoin_signed_graph_pseudo_labels
[2026-05-17 09:22:18]   nodes: 3783
[2026-05-17 09:22:18]   anomalies: 383

RUNNING: dataset=Bitcoin_Alpha, ablation=attr_only, model=LogisticRegression
[2026-05-17 09:22:18] ====================================================================================================
[2026-05-17 09:22:18] RUN dataset=Bitcoin_Alpha, ablation=attr_only, model=LogisticRegression
[2026-05-17 09:22:18] ====================================================================================================
[2026-05-17 09:22:18] X shape: (3783

In [14]:
# ============================================================
# Collect summary after every run / resume
# Reads all JSON files from results_json
# ============================================================

def collect_results() -> pd.DataFrame:
    rows = []

    for path in sorted(JSON_DIR.glob("*.json")):
        try:
            r = load_json(path)
        except Exception as e:
            print("[WARN] bad json:", path, repr(e))
            continue

        row = {
            "dataset": r.get("dataset"),
            "dataset_kind": r.get("dataset_kind"),
            "ablation": r.get("ablation"),
            "model": r.get("model"),
            "status": r.get("status"),
            "n_nodes": r.get("n_nodes"),
            "n_features": r.get("n_features"),
            "n_train": r.get("n_train"),
            "n_test": r.get("n_test"),
            "n_anomalies_total": r.get("n_anomalies_total"),
            "n_anomalies_train": r.get("n_anomalies_train"),
            "n_anomalies_test": r.get("n_anomalies_test"),
            "runtime_sec": r.get("runtime_sec"),
            "json_file": str(path),
        }

        metrics = r.get("metrics", {})
        for k, v in metrics.items():
            row[k] = v

        rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        sort_cols = [c for c in ["dataset", "ablation", "model"] if c in df.columns]
        df = df.sort_values(sort_cols).reset_index(drop=True)

    return df


summary_df = collect_results()

csv_path = SUMMARY_DIR / "results_summary.csv"
xlsx_path = SUMMARY_DIR / "results_summary.xlsx"

summary_df.to_csv(csv_path, index=False)

try:
    summary_df.to_excel(xlsx_path, index=False)
except Exception as e:
    print("[WARN] Excel save failed:", repr(e))

print("Saved:", csv_path)
print("Saved:", xlsx_path)
display(summary_df)

[WARN] Excel save failed: ModuleNotFoundError("No module named 'openpyxl'")
Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\ablation_trivial_baselines_outputs\summary\results_summary.csv
Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\ablation_trivial_baselines_outputs\summary\results_summary.xlsx


,dataset,dataset_kind,ablation,model,status,n_nodes,n_features,n_train,n_test,n_anomalies_total,...,k_at_1pct,hits_at_1pct,precision_at_k_at_1pct,recall_at_k_at_1pct,hits_at_k_at_1pct,k_at_5pct,hits_at_5pct,precision_at_k_at_5pct,recall_at_k_at_5pct,hits_at_k_at_5pct
0,Bitcoin_Alpha,bitcoin_signed_graph_pseudo_labels,attr_only,ExtraTrees,ok,3783,1,3026,757,383,...,7,1,0.142857,0.013158,1.0,37,4,0.108108,0.052632,1.0
1,Bitcoin_Alpha,bitcoin_signed_graph_pseudo_labels,attr_only,GradientBoosting,ok,3783,1,3026,757,383,...,7,1,0.142857,0.013158,1.0,37,4,0.108108,0.052632,1.0
2,Bitcoin_Alpha,bitcoin_signed_graph_pseudo_labels,attr_only,IsolationForest,ok,3783,1,3026,757,383,...,7,1,0.142857,0.013158,1.0,37,4,0.108108,0.052632,1.0
3,Bitcoin_Alpha,bitcoin_signed_graph_pseudo_labels,attr_only,LogisticRegression,ok,3783,1,3026,757,383,...,7,1,0.142857,0.013158,1.0,37,4,0.108108,0.052632,1.0
4,Bitcoin_Alpha,bitcoin_signed_graph_pseudo_labels,attr_only,RandomForest,ok,3783,1,3026,757,383,...,7,1,0.142857,0.013158,1.0,37,4,0.108108,0.052632,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,FraudYelp,dgl_fraud_heterograph,struct_only,ExtraTrees,ok,45954,21,36762,9192,6677,...,91,48,0.527473,0.036923,1.0,459,139,0.302832,0.106923,1.0
96,FraudYelp,dgl_fraud_heterograph,struct_only,GradientBoosting,ok,45954,21,36762,9192,6677,...,91,41,0.450549,0.031538,1.0,459,125,0.272331,0.096154,1.0
97,FraudYelp,dgl_fraud_heterograph,struct_only,IsolationForest,ok,45954,21,36762,9192,6677,...,91,21,0.230769,0.016154,1.0,459,87,0.189542,0.066923,1.0
98,FraudYelp,dgl_fraud_heterograph,struct_only,LogisticRegression,ok,45954,21,36762,9192,6677,...,91,27,0.296703,0.020769,1.0,459,107,0.233115,0.082308,1.0


In [15]:
# ============================================================
# Main result tables
# ============================================================

summary_df = collect_results()

ok_df = summary_df[summary_df["status"] == "ok"].copy()

print("=" * 120)
print("BEST BY ROC-AUC")
print("=" * 120)

if "roc_auc" in ok_df.columns:
    best_roc = (
        ok_df
        .sort_values(["dataset", "roc_auc"], ascending=[True, False])
        .groupby("dataset")
        .head(10)
        .reset_index(drop=True)
    )
    display(best_roc[[
        "dataset", "ablation", "model", "roc_auc", "pr_auc",
        "precision_at_k_at_pos", "recall_at_k_at_pos",
        "n_features", "runtime_sec"
    ]])

print("=" * 120)
print("BEST BY PR-AUC")
print("=" * 120)

if "pr_auc" in ok_df.columns:
    best_pr = (
        ok_df
        .sort_values(["dataset", "pr_auc"], ascending=[True, False])
        .groupby("dataset")
        .head(10)
        .reset_index(drop=True)
    )
    display(best_pr[[
        "dataset", "ablation", "model", "roc_auc", "pr_auc",
        "precision_at_k_at_pos", "recall_at_k_at_pos",
        "n_features", "runtime_sec"
    ]])

BEST BY ROC-AUC


,dataset,ablation,model,roc_auc,pr_auc,precision_at_k_at_pos,recall_at_k_at_pos,n_features,runtime_sec
0,Bitcoin_Alpha,full,GradientBoosting,1.000000,1.000000,1.000000,1.000000,21,0.847763
1,Bitcoin_Alpha,full,RandomForest,1.000000,1.000000,1.000000,1.000000,21,1.240602
2,Bitcoin_Alpha,no_degree_struct,ExtraTrees,1.000000,1.000000,1.000000,1.000000,12,0.596330
3,Bitcoin_Alpha,no_degree_struct,GradientBoosting,1.000000,1.000000,1.000000,1.000000,12,0.420453
4,Bitcoin_Alpha,no_degree_struct,RandomForest,1.000000,1.000000,1.000000,1.000000,12,1.193460
5,Bitcoin_Alpha,struct_only,GradientBoosting,1.000000,1.000000,1.000000,1.000000,21,1.018792
6,Bitcoin_Alpha,struct_only,RandomForest,1.000000,1.000000,1.000000,1.000000,21,1.198155
7,Bitcoin_Alpha,full,ExtraTrees,0.999961,0.999663,0.986842,0.986842,21,0.666879
8,Bitcoin_Alpha,struct_only,ExtraTrees,0.999961,0.999663,0.986842,0.986842,21,0.706341
9,Bitcoin_Alpha,full,LogisticRegression,0.999884,0.998956,0.986842,0.986842,21,0.033288


BEST BY PR-AUC


,dataset,ablation,model,roc_auc,pr_auc,precision_at_k_at_pos,recall_at_k_at_pos,n_features,runtime_sec
0,Bitcoin_Alpha,full,GradientBoosting,1.000000,1.000000,1.000000,1.000000,21,0.847763
1,Bitcoin_Alpha,full,RandomForest,1.000000,1.000000,1.000000,1.000000,21,1.240602
2,Bitcoin_Alpha,no_degree_struct,GradientBoosting,1.000000,1.000000,1.000000,1.000000,12,0.420453
3,Bitcoin_Alpha,no_degree_struct,RandomForest,1.000000,1.000000,1.000000,1.000000,12,1.193460
4,Bitcoin_Alpha,struct_only,GradientBoosting,1.000000,1.000000,1.000000,1.000000,21,1.018792
5,Bitcoin_Alpha,struct_only,RandomForest,1.000000,1.000000,1.000000,1.000000,21,1.198155
6,Bitcoin_Alpha,no_degree_struct,ExtraTrees,1.000000,1.000000,1.000000,1.000000,12,0.596330
7,Bitcoin_Alpha,full,ExtraTrees,0.999961,0.999663,0.986842,0.986842,21,0.666879
8,Bitcoin_Alpha,struct_only,ExtraTrees,0.999961,0.999663,0.986842,0.986842,21,0.706341
9,Bitcoin_Alpha,full,LogisticRegression,0.999884,0.998956,0.986842,0.986842,21,0.033288


In [16]:
# ============================================================
# Shortcut sensitivity analysis
#
# For each dataset:
# - best structural/degree trivial baseline
# - best full baseline
# - gap full - trivial
# ============================================================

summary_df = collect_results()
ok_df = summary_df[summary_df["status"] == "ok"].copy()

if not ok_df.empty and "roc_auc" in ok_df.columns:
    shortcut_rows = []

    for dataset, g in ok_df.groupby("dataset"):
        g_metric = g.dropna(subset=["roc_auc"]).copy()

        if g_metric.empty:
            continue

        trivial = g_metric[g_metric["ablation"].isin(["struct_only", "degree_only", "no_degree_struct"])]
        full = g_metric[g_metric["ablation"].isin(["full"])]
        attr = g_metric[g_metric["ablation"].isin(["attr_only"])]

        best_trivial = trivial.sort_values("roc_auc", ascending=False).head(1)
        best_full = full.sort_values("roc_auc", ascending=False).head(1)
        best_attr = attr.sort_values("roc_auc", ascending=False).head(1)

        row = {"dataset": dataset}

        if not best_trivial.empty:
            bt = best_trivial.iloc[0]
            row["best_trivial_ablation"] = bt["ablation"]
            row["best_trivial_model"] = bt["model"]
            row["best_trivial_roc_auc"] = bt["roc_auc"]
            row["best_trivial_pr_auc"] = bt.get("pr_auc")

        if not best_full.empty:
            bf = best_full.iloc[0]
            row["best_full_model"] = bf["model"]
            row["best_full_roc_auc"] = bf["roc_auc"]
            row["best_full_pr_auc"] = bf.get("pr_auc")

        if not best_attr.empty:
            ba = best_attr.iloc[0]
            row["best_attr_model"] = ba["model"]
            row["best_attr_roc_auc"] = ba["roc_auc"]
            row["best_attr_pr_auc"] = ba.get("pr_auc")

        if "best_trivial_roc_auc" in row and "best_full_roc_auc" in row:
            row["full_minus_trivial_roc_auc"] = row["best_full_roc_auc"] - row["best_trivial_roc_auc"]

        if "best_attr_roc_auc" in row and "best_trivial_roc_auc" in row:
            row["attr_minus_trivial_roc_auc"] = row["best_attr_roc_auc"] - row["best_trivial_roc_auc"]

        shortcut_rows.append(row)

    shortcut_df = pd.DataFrame(shortcut_rows)
    shortcut_csv = SUMMARY_DIR / "shortcut_sensitivity_summary.csv"
    shortcut_xlsx = SUMMARY_DIR / "shortcut_sensitivity_summary.xlsx"

    shortcut_df.to_csv(shortcut_csv, index=False)
    try:
        shortcut_df.to_excel(shortcut_xlsx, index=False)
    except Exception as e:
        print("[WARN] Excel save failed:", repr(e))

    print("Saved:", shortcut_csv)
    print("Saved:", shortcut_xlsx)
    display(shortcut_df)
else:
    print("No OK results available yet.")

[WARN] Excel save failed: ModuleNotFoundError("No module named 'openpyxl'")
Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\ablation_trivial_baselines_outputs\summary\shortcut_sensitivity_summary.csv
Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\ablation_trivial_baselines_outputs\summary\shortcut_sensitivity_summary.xlsx


,dataset,best_trivial_ablation,best_trivial_model,best_trivial_roc_auc,best_trivial_pr_auc,best_full_model,best_full_roc_auc,best_full_pr_auc,best_attr_model,best_attr_roc_auc,best_attr_pr_auc,full_minus_trivial_roc_auc,attr_minus_trivial_roc_auc
0,Bitcoin_Alpha,no_degree_struct,ExtraTrees,1.000000,1.000000,GradientBoosting,1.000000,1.000000,ExtraTrees,0.500000,0.100396,0.000000,-0.500000
1,Bitcoin_OTC,struct_only,LogisticRegression,0.999944,0.999510,LogisticRegression,0.999944,0.999510,ExtraTrees,0.500000,0.100255,0.000000,-0.499944
2,FraudAmazon,degree_only,GradientBoosting,0.815115,0.382211,GradientBoosting,0.987361,0.931436,GradientBoosting,0.986112,0.927526,0.172246,0.170997
3,FraudYelp,struct_only,ExtraTrees,0.579907,0.217710,RandomForest,0.936446,0.774882,RandomForest,0.937430,0.783490,0.356539,0.357523


In [17]:
# ============================================================
# Create readable TXT report
# ============================================================

summary_df = collect_results()
ok_df = summary_df[summary_df["status"] == "ok"].copy()

report_path = SUMMARY_DIR / "method_results_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("Ablation-Driven Evaluation of Trivial Feature Baselines for Realistic Anomaly Detection\n")
    f.write("=" * 100 + "\n\n")

    f.write("Datasets used:\n")
    for ds in datasets:
        f.write(f"- {ds['name']}: kind={ds.get('kind')}, nodes={len(ds['y'])}, anomalies={int(np.sum(ds['y']))}\n")
        f.write(f"  notes: {ds.get('notes')}\n")
    f.write("\n")

    f.write("Ablations:\n")
    for a in ABLATIONS:
        f.write(f"- {a}\n")
    f.write("\n")

    f.write("Models:\n")
    for m in ALL_MODELS:
        f.write(f"- {m}\n")
    f.write("\n")

    if not ok_df.empty:
        f.write("Best results by dataset using ROC-AUC:\n")
        f.write("-" * 100 + "\n")

        for dataset, g in ok_df.groupby("dataset"):
            f.write(f"\nDataset: {dataset}\n")
            g2 = g.dropna(subset=["roc_auc"]).sort_values("roc_auc", ascending=False).head(10)

            for _, r in g2.iterrows():
                f.write(
                    f"  {r['ablation']:20s} | {r['model']:20s} | "
                    f"ROC-AUC={r.get('roc_auc')} | PR-AUC={r.get('pr_auc')} | "
                    f"P@K={r.get('precision_at_k_at_pos')} | R@K={r.get('recall_at_k_at_pos')}\n"
                )

        f.write("\n\nShortcut interpretation:\n")
        f.write("-" * 100 + "\n")
        f.write(
            "If struct_only or degree_only performs close to full, the benchmark is shortcut-sensitive.\n"
            "If full substantially outperforms structural features, attributes or richer interactions matter.\n"
            "If Bitcoin pseudo-labels are solved very well by degree/signed features, this confirms that the pseudo-label task is intentionally shortcut-heavy.\n"
        )

print("Saved report:", report_path)

with open(report_path, "r", encoding="utf-8") as f:
    print(f.read()[:5000])

Saved report: C:\Users\ivan\WORK\workshops\ICDM\Ablation\ablation_trivial_baselines_outputs\summary\method_results_report.txt
Ablation-Driven Evaluation of Trivial Feature Baselines for Realistic Anomaly Detection

Datasets used:
- Bitcoin_Alpha: kind=bitcoin_signed_graph_pseudo_labels, nodes=3783, anomalies=383
  notes: Pseudo-labels from negative trust statistics; use as diagnostic stress-test, not real fraud ground truth.
- Bitcoin_OTC: kind=bitcoin_signed_graph_pseudo_labels, nodes=5881, anomalies=591
  notes: Pseudo-labels from negative trust statistics; use as diagnostic stress-test, not real fraud ground truth.
- FraudYelp: kind=dgl_fraud_heterograph, nodes=45954, anomalies=6677
  notes: Real labels from DGL fraud benchmark.
- FraudAmazon: kind=dgl_fraud_heterograph, nodes=11944, anomalies=821
  notes: Real labels from DGL fraud benchmark.

Ablations:
- attr_only
- struct_only
- full
- degree_only
- no_degree_struct

Models:
- LogisticRegression
- RandomForest
- ExtraTrees
- Gra

In [18]:
from pathlib import Path
import json
import pandas as pd

OUT_ROOT = Path.cwd() / "ablation_trivial_baselines_outputs"
JSON_DIR = OUT_ROOT / "results_json"
SUMMARY_DIR = OUT_ROOT / "summary"

rows = []

for p in sorted(JSON_DIR.glob("*.json")):
    try:
        with open(p, "r", encoding="utf-8") as f:
            r = json.load(f)
    except Exception as e:
        rows.append({
            "file": p.name,
            "status": "bad_json",
            "error": repr(e),
        })
        continue

    row = {
        "file": p.name,
        "dataset": r.get("dataset"),
        "ablation": r.get("ablation"),
        "model": r.get("model"),
        "status": r.get("status"),
        "error": r.get("error"),
        "roc_auc": r.get("metrics", {}).get("roc_auc"),
        "pr_auc": r.get("metrics", {}).get("pr_auc"),
        "precision_at_k_at_pos": r.get("metrics", {}).get("precision_at_k_at_pos"),
        "recall_at_k_at_pos": r.get("metrics", {}).get("recall_at_k_at_pos"),
        "runtime_sec": r.get("runtime_sec"),
    }
    rows.append(row)

df = pd.DataFrame(rows)

print("Total JSON files:", len(df))
display(df)

print("\nStatus counts:")
display(df.groupby(["dataset", "status"]).size().reset_index(name="count"))

print("\nDataset counts:")
display(df.groupby("dataset").size().reset_index(name="n_runs"))

print("\nErrors:")
display(df[df["status"] != "ok"][["dataset", "ablation", "model", "status", "error", "file"]])

Total JSON files: 100


,file,dataset,ablation,model,status,error,roc_auc,pr_auc,precision_at_k_at_pos,recall_at_k_at_pos,runtime_sec
0,Bitcoin_Alpha__attr_only__ExtraTrees.json,Bitcoin_Alpha,attr_only,ExtraTrees,ok,None,0.500000,0.100396,0.092105,0.092105,0.559704
1,Bitcoin_Alpha__attr_only__GradientBoosting.json,Bitcoin_Alpha,attr_only,GradientBoosting,ok,None,0.500000,0.100396,0.092105,0.092105,0.154839
2,Bitcoin_Alpha__attr_only__IsolationForest.json,Bitcoin_Alpha,attr_only,IsolationForest,ok,None,0.500000,0.100396,0.092105,0.092105,0.643246
3,Bitcoin_Alpha__attr_only__LogisticRegression.json,Bitcoin_Alpha,attr_only,LogisticRegression,ok,None,0.500000,0.100396,0.092105,0.092105,0.278808
4,Bitcoin_Alpha__attr_only__RandomForest.json,Bitcoin_Alpha,attr_only,RandomForest,ok,None,0.500000,0.100396,0.092105,0.092105,3.158542
...,...,...,...,...,...,...,...,...,...,...,...
95,FraudYelp__struct_only__ExtraTrees.json,FraudYelp,struct_only,ExtraTrees,ok,None,0.579907,0.217710,0.232308,0.232308,3.144868
96,FraudYelp__struct_only__GradientBoosting.json,FraudYelp,struct_only,GradientBoosting,ok,None,0.575102,0.201293,0.230769,0.230769,27.211545
97,FraudYelp__struct_only__IsolationForest.json,FraudYelp,struct_only,IsolationForest,ok,None,0.545602,0.167585,0.190000,0.190000,1.665540
98,FraudYelp__struct_only__LogisticRegression.json,FraudYelp,struct_only,LogisticRegression,ok,None,0.549035,0.173848,0.197692,0.197692,0.173004



Status counts:


,dataset,status,count
0,Bitcoin_Alpha,ok,25
1,Bitcoin_OTC,ok,25
2,FraudAmazon,ok,25
3,FraudYelp,ok,25



Dataset counts:


,dataset,n_runs
0,Bitcoin_Alpha,25
1,Bitcoin_OTC,25
2,FraudAmazon,25
3,FraudYelp,25



Errors:


,dataset,ablation,model,status,error,file


In [19]:
ok = df[df["status"] == "ok"].copy()

metric = "roc_auc"

best = (
    ok
    .dropna(subset=[metric])
    .sort_values(["dataset", metric], ascending=[True, False])
    .groupby("dataset")
    .head(10)
    .reset_index(drop=True)
)

display(best[[
    "dataset",
    "ablation",
    "model",
    "roc_auc",
    "pr_auc",
    "precision_at_k_at_pos",
    "recall_at_k_at_pos",
    "runtime_sec",
]])

,dataset,ablation,model,roc_auc,pr_auc,precision_at_k_at_pos,recall_at_k_at_pos,runtime_sec
0,Bitcoin_Alpha,full,GradientBoosting,1.000000,1.000000,1.000000,1.000000,0.847763
1,Bitcoin_Alpha,full,RandomForest,1.000000,1.000000,1.000000,1.000000,1.240602
2,Bitcoin_Alpha,no_degree_struct,ExtraTrees,1.000000,1.000000,1.000000,1.000000,0.596330
3,Bitcoin_Alpha,no_degree_struct,GradientBoosting,1.000000,1.000000,1.000000,1.000000,0.420453
4,Bitcoin_Alpha,no_degree_struct,RandomForest,1.000000,1.000000,1.000000,1.000000,1.193460
5,Bitcoin_Alpha,struct_only,GradientBoosting,1.000000,1.000000,1.000000,1.000000,1.018792
6,Bitcoin_Alpha,struct_only,RandomForest,1.000000,1.000000,1.000000,1.000000,1.198155
7,Bitcoin_Alpha,full,ExtraTrees,0.999961,0.999663,0.986842,0.986842,0.666879
8,Bitcoin_Alpha,struct_only,ExtraTrees,0.999961,0.999663,0.986842,0.986842,0.706341
9,Bitcoin_Alpha,full,LogisticRegression,0.999884,0.998956,0.986842,0.986842,0.033288


In [20]:
ok = df[df["status"] == "ok"].copy()

pivot = (
    ok
    .groupby(["dataset", "ablation"])["roc_auc"]
    .max()
    .reset_index()
    .pivot(index="dataset", columns="ablation", values="roc_auc")
    .reset_index()
)

for col in ["attr_only", "struct_only", "full", "degree_only", "no_degree_struct"]:
    if col not in pivot.columns:
        pivot[col] = None

pivot["full_minus_attr"] = pivot["full"] - pivot["attr_only"]
pivot["full_minus_struct"] = pivot["full"] - pivot["struct_only"]
pivot["struct_minus_attr"] = pivot["struct_only"] - pivot["attr_only"]

display(pivot)

out = SUMMARY_DIR / "ablation_gap_table.csv"
pivot.to_csv(out, index=False)
print("Saved:", out)

ablation,dataset,attr_only,degree_only,full,no_degree_struct,struct_only,full_minus_attr,full_minus_struct,struct_minus_attr
0,Bitcoin_Alpha,0.500000,0.831807,1.000000,1.000000,1.000000,0.500000,0.000000,0.500000
1,Bitcoin_OTC,0.500000,0.876618,0.999944,0.999872,0.999944,0.499944,0.000000,0.499944
2,FraudAmazon,0.986112,0.815115,0.987361,0.765967,0.814943,0.001249,0.172418,-0.171169
3,FraudYelp,0.937430,0.579787,0.936446,0.540358,0.579907,-0.000984,0.356539,-0.357523


Saved: C:\Users\ivan\WORK\workshops\ICDM\Ablation\ablation_trivial_baselines_outputs\summary\ablation_gap_table.csv
